# Data Cleaning Pipeline 2v2

Notebook unificado para ejecutar en orden:
1. LimpiezaDatos
2. ColumnDeleter
3. CambiosAVG

Este flujo usa DataFrames en memoria entre secciones y solo guarda el CSV final.

## Section 0 


In [8]:
import os
import re
import sys
import json
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 200)

# =========================
# CONFIG GLOBAL
# =========================
DATASETS_DIR = "datasets"
os.makedirs(DATASETS_DIR, exist_ok=True)

CSV_IN = os.path.join(DATASETS_DIR, "replays_2v2_10000_full.csv")
CSV_OUT = os.path.join(DATASETS_DIR, "replays_subset_with_time_percentages.csv")

THRESHOLD_NULL_RATIO = 0.0001
TREAT_EMPTY_STRINGS_AS_NULL = True
KEEP_COLS = []
SHOW_EXAMPLES = 5
MAX_PRINT_CANDIDATES = 400  # 0 = imprimir todas
DROP_CAMERA_COLUMNS = True
DROP_MATCHES_SHORTER_THAN_SECONDS = 3

# Persistencia de decisiones de la seccion 2.5
DECISIONS_25_FILE = os.path.join(DATASETS_DIR, "section_2_5_decisions.json")

DURATION_CANDIDATES = [
    "duration",
    "duration_seconds",
    "game.duration",
    "game.duration_seconds",
    "replay.duration",
    "replay.duration_seconds",
    "match.duration",
    "match.duration_seconds",
]

print(f"CSV_IN:  {CSV_IN}")
print(f"CSV_OUT: {CSV_OUT}")
print(f"DECISIONS_25_FILE: {DECISIONS_25_FILE}")

CSV_IN:  datasets/replays_2v2_10000_full.csv
CSV_OUT: datasets/replays_subset_with_time_percentages.csv
DECISIONS_25_FILE: datasets/section_2_5_decisions.json


In [9]:
# =========================
# HELPERS COMPARTIDOS
# =========================
def read_csv_safely(path: str) -> pd.DataFrame:
    try:
        return pd.read_csv(path, low_memory=False)
    except UnicodeDecodeError:
        return pd.read_csv(path, encoding="latin-1", low_memory=False)


def to_numeric(series: pd.Series) -> pd.Series:
    return pd.to_numeric(series, errors="coerce")


def is_numeric_dtype(dtype) -> bool:
    return pd.api.types.is_numeric_dtype(dtype)


def is_bool_dtype(dtype) -> bool:
    return pd.api.types.is_bool_dtype(dtype)


def shorten(x, n=180):
    s = str(x)
    return s if len(s) <= n else s[:n] + "..."


def print_examples(series: pd.Series, n=5):
    non_null = series.dropna()
    if non_null.empty:
        print("  (no hay valores no nulos para mostrar)")
        return
    for i, v in enumerate(non_null.head(n).tolist(), start=1):
        print(f"  {i}. {shorten(v)}")


def ask_action(col: str, null_pct: float, dtype: str) -> str:
    while True:
        print("\n----------------------------------------")
        print(f"Columna candidata: {col}")
        print(f"  - null_pct: {null_pct:.2f}%")
        print(f"  - dtype: {dtype}")
        ans = input("Accion: [d]=eliminar, [k]=mantener y rellenar nulls, [s]=ver ejemplos, [q]=salir: ").strip().lower()
        if ans in {"d", "k", "s", "q"}:
            return ans
        print("Entrada no valida. Usa d/k/s/q.")


def ask_imputation_numeric(col: str, series: pd.Series) -> tuple[str, object]:
    while True:
        print("\nRelleno de nulls (columna numerica):")
        print("  [1] media")
        print("  [2] mediana")
        print("  [3] moda (valor mas frecuente)")
        print("  [4] constante (lo indicas tu)")
        print("  [5] cero")
        print("  [6] no rellenar (dejar nulls)")
        choice = input("Elige 1/2/3/4/5/6: ").strip()

        s = series.dropna()
        if choice == "1":
            if s.empty:
                print("No hay datos no nulos para calcular media. Elige constante.")
                continue
            return ("mean", float(s.mean()))
        if choice == "2":
            if s.empty:
                print("No hay datos no nulos para calcular mediana. Elige constante.")
                continue
            return ("median", float(s.median()))
        if choice == "3":
            if s.empty:
                print("No hay datos no nulos para calcular moda. Elige constante.")
                continue
            mode = s.mode(dropna=True)
            if mode.empty:
                print("No se pudo calcular moda. Elige constante.")
                continue
            try:
                return ("mode", float(mode.iloc[0]))
            except Exception:
                return ("mode", mode.iloc[0])
        if choice == "4":
            raw = input(f"Valor constante para '{col}' (ej: 0, -1, 3.14): ").strip()
            try:
                val = float(raw)
            except ValueError:
                print("Ese valor no parece numerico. Intenta de nuevo.")
                continue
            return ("constant", val)
        if choice == "5":
            return ("zero", 0.0)
        if choice == "6":
            return ("none", None)
        print("Opcion no valida.")


def ask_imputation_bool(col: str, series: pd.Series) -> tuple[str, object]:
    while True:
        print("\nRelleno de nulls (columna booleana):")
        print("  [1] True")
        print("  [2] False")
        print("  [3] moda (valor mas frecuente)")
        print("  [4] no rellenar (dejar nulls)")
        choice = input("Elige 1/2/3/4: ").strip()

        s = series.dropna()
        if choice == "1":
            return ("true", True)
        if choice == "2":
            return ("false", False)
        if choice == "3":
            if s.empty:
                print("No hay datos no nulos para calcular moda. Elige True/False.")
                continue
            mode = s.mode(dropna=True)
            if mode.empty:
                print("No se pudo calcular moda. Elige True/False.")
                continue
            return ("mode", bool(mode.iloc[0]))
        if choice == "4":
            return ("none", None)
        print("Opcion no valida.")


def ask_imputation_categorical(col: str, series: pd.Series) -> tuple[str, object]:
    while True:
        print("\nRelleno de nulls (columna no numerica):")
        print("  [1] moda (valor mas frecuente)")
        print("  [2] constante (lo indicas tu, texto)")
        print("  [3] 'MISSING' (texto literal)")
        print("  [4] cadena vacia ''")
        print("  [5] boolean True")
        print("  [6] boolean False")
        print("  [7] no rellenar (dejar nulls)")
        choice = input("Elige 1/2/3/4/5/6/7: ").strip()

        s = series.dropna()
        if choice == "1":
            if s.empty:
                print("No hay datos no nulos para calcular moda. Elige constante.")
                continue
            mode = s.mode(dropna=True)
            if mode.empty:
                print("No se pudo calcular moda. Elige constante.")
                continue
            return ("mode", str(mode.iloc[0]))
        if choice == "2":
            val = input(f"Valor constante (texto) para '{col}': ").strip()
            return ("constant", val)
        if choice == "3":
            return ("missing_literal", "MISSING")
        if choice == "4":
            return ("empty_string", "")
        if choice == "5":
            return ("true", True)
        if choice == "6":
            return ("false", False)
        if choice == "7":
            return ("none", None)
        print("Opcion no valida.")


def _to_python_scalar(value):
    if value is None:
        return None
    if pd.isna(value):
        return None
    if isinstance(value, np.generic):
        return value.item()
    return value


def load_decisions(path: str) -> dict:
    if not os.path.exists(path):
        return {}
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
    if not isinstance(data, dict):
        return {}
    return data


def save_decisions(path: str, decisions: dict) -> None:
    serializable = {}
    for col, payload in decisions.items():
        if not isinstance(payload, dict):
            continue
        action = payload.get("action")
        if action == "drop":
            serializable[col] = {"action": "drop"}
        elif action == "impute":
            serializable[col] = {
                "action": "impute",
                "strategy": payload.get("strategy", "none"),
                "value": _to_python_scalar(payload.get("value")),
            }
    with open(path, "w", encoding="utf-8") as f:
        json.dump(serializable, f, ensure_ascii=True, indent=2)


def apply_saved_decisions(df_in: pd.DataFrame, decisions: dict) -> tuple[pd.DataFrame, list[str], dict[str, tuple[str, object]]]:
    dropped_cols = []
    imputed_cols = {}
    df_out = df_in.copy()

    for col, payload in decisions.items():
        if col not in df_out.columns:
            continue
        action = payload.get("action")

        if action == "drop":
            dropped_cols.append(col)
            continue

        if action == "impute":
            strat = payload.get("strategy", "none")
            val = payload.get("value")
            if strat != "none":
                df_out[col] = df_out[col].fillna(val)
            preview = shorten(val, 60) if isinstance(val, str) else val
            imputed_cols[col] = (strat, preview)

    if dropped_cols:
        cols_to_drop = [c for c in dropped_cols if c in df_out.columns]
        df_out = df_out.drop(columns=cols_to_drop)

    return df_out, dropped_cols, imputed_cols


def is_camera_col(col: str) -> bool:
    c = col.lower()

    camera_keywords = [
        "camera", "camera_settings", "cam_settings", "camerasettings", "camsettings"
    ]
    if any(k in c for k in camera_keywords):
        return True

    cam_params = ["fov", "distance", "height", "angle", "stiffness", "swivel", "transition"]
    if any(p in c for p in cam_params) and ("player" in c or "players" in c) and ("setting" in c or "settings" in c):
        return True

    return False


def find_duration_column(df: pd.DataFrame) -> str | None:
    for c in DURATION_CANDIDATES:
        if c in df.columns:
            return c
    duration_like = [c for c in df.columns if "duration" in c.lower()]
    if duration_like:
        for c in duration_like:
            if "second" in c.lower():
                return c
        return duration_like[0]
    return None


def rank_name_without_division(value):
    if value is None or pd.isna(value):
        return pd.NA
    s = str(value).strip()
    if not s:
        return pd.NA
    s = re.sub(r"\s+Division\s+\d+\s*$", "", s, flags=re.IGNORECASE)
    return s.strip()


def add_grouped_game_rank_columns(df_in: pd.DataFrame) -> pd.DataFrame:
    df_out = df_in.copy()
    mapping = [
        ("min_rank.name", "min.game.rank"),
        ("max_rank.name", "max.game.rank"),
    ]

    for src_col, dst_col in mapping:
        if src_col not in df_out.columns:
            print(f"[WARN] No existe columna fuente para rank agrupado: {src_col}")
            continue
        df_out[dst_col] = df_out[src_col].apply(rank_name_without_division).astype("string")

    return df_out

## Section 1 - Limpieza automatica (en memoria)

Esta seccion carga el CSV inicial y aplica limpieza automatica (camara y duracion).

In [10]:
if not os.path.exists(CSV_IN):
    raise FileNotFoundError(f"No existe el archivo: {CSV_IN}")

df = read_csv_safely(CSV_IN)

if TREAT_EMPTY_STRINGS_AS_NULL:
    df = df.replace(r"^\s*$", pd.NA, regex=True)

rows_before_all = len(df)
cols_before_all = df.shape[1]

print("========================================")
print("Carga inicial")
print("========================================")
print(f"Filas iniciales:    {rows_before_all:,}")
print(f"Columnas iniciales: {cols_before_all:,}")

Carga inicial
Filas iniciales:    10,000
Columnas iniciales: 561


In [11]:
# 0) Eliminacion automatica de columnas de camara
if DROP_CAMERA_COLUMNS:
    camera_cols = [c for c in df.columns if is_camera_col(c)]
    print("\n========================================")
    print("Eliminacion automatica: columnas de camara")
    print("========================================")
    if camera_cols:
        print(f"Columnas detectadas: {len(camera_cols):,}")
        for c in camera_cols:
            print(f"- {c}")
        df = df.drop(columns=camera_cols)
    else:
        print("No se detectaron columnas de camara con el patron actual.")

# 1) Eliminacion automatica de partidas cortas
if DROP_MATCHES_SHORTER_THAN_SECONDS is not None:
    print("\n========================================")
    print(f"Eliminacion automatica: partidas < {DROP_MATCHES_SHORTER_THAN_SECONDS}s")
    print("========================================")
    dur_col = find_duration_column(df)
    if dur_col is None:
        print("No se encontro columna de duracion. No se eliminan filas por duracion.")
        duration_like = [c for c in df.columns if "duration" in c.lower()]
        if duration_like:
            print("Columnas con 'duration':")
            for c in duration_like[:50]:
                print(f"- {c}")
            if len(duration_like) > 50:
                print(f"... ({len(duration_like) - 50} mas)")
        else:
            print("(ninguna)")
    else:
        dur = pd.to_numeric(df[dur_col], errors="coerce")
        mask_short = dur.notna() & (dur < DROP_MATCHES_SHORTER_THAN_SECONDS)
        removed_short = int(mask_short.sum())
        df = df.loc[~mask_short].copy()

        print(f"Columna usada: {dur_col}")
        print(f"Filas eliminadas por duracion: {removed_short:,}")
        print(f"Filas restantes: {len(df):,}")

print("\n========================================")
print("Section 1 completada (limpieza automatica)")
print("========================================")
print(f"Filas tras limpieza automatica:    {len(df):,}")
print(f"Columnas tras limpieza automatica: {df.shape[1]:,}")

df_stage1 = df.copy()


Eliminacion automatica: columnas de camara
Columnas detectadas: 28
- blue.players.0.camera.distance
- blue.players.0.camera.fov
- blue.players.0.camera.height
- blue.players.0.camera.pitch
- blue.players.0.camera.stiffness
- blue.players.0.camera.swivel_speed
- blue.players.0.camera.transition_speed
- blue.players.1.camera.distance
- blue.players.1.camera.fov
- blue.players.1.camera.height
- blue.players.1.camera.pitch
- blue.players.1.camera.stiffness
- blue.players.1.camera.swivel_speed
- blue.players.1.camera.transition_speed
- orange.players.0.camera.distance
- orange.players.0.camera.fov
- orange.players.0.camera.height
- orange.players.0.camera.pitch
- orange.players.0.camera.stiffness
- orange.players.0.camera.swivel_speed
- orange.players.0.camera.transition_speed
- orange.players.1.camera.distance
- orange.players.1.camera.fov
- orange.players.1.camera.height
- orange.players.1.camera.pitch
- orange.players.1.camera.stiffness
- orange.players.1.camera.swivel_speed
- orange.pl

## Section 2 - ColumnDeleter (en memoria)

Primero se aplica la lista corregida de columnas para reducir trabajo innecesario antes de la imputacion interactiva.

In [12]:
def build_keep_columns() -> list[str]:
    keep = set()

    keep.update([
        "match_guid",
        "duration",
        "overtime",
        "overtime_seconds",
        "min.game.rank",
        "max.game.rank",
        "server.name",
        "server.region",
    ])

    teams = ["blue", "orange"]
    team_ball = ["possession_time", "time_in_side"]
    for t in teams:
        for f in team_ball:
            keep.add(f"{t}.stats.ball.{f}")

    player_core = [
        "shots", "shots_against", "goals", "goals_against", "saves", "assists", "score", "shooting_percentage"
    ]

    player_boost = [
        "bpm", "bcpm", "avg_amount",
        "amount_collected", "amount_stolen",
        "amount_collected_big", "amount_stolen_big",
        "amount_collected_small", "amount_stolen_small",
        "amount_overfill", "amount_overfill_stolen",
        "amount_used_while_supersonic",
        "percent_zero_boost", "percent_full_boost",
        "percent_boost_0_25", "percent_boost_25_50", "percent_boost_50_75", "percent_boost_75_100",
    ]

    player_movement = [
        "avg_speed", "total_distance",
        "time_supersonic_speed", "time_boost_speed", "time_slow_speed",
        "time_ground", "time_low_air", "time_high_air",
        "time_powerslide", "count_powerslide",
        "avg_powerslide_duration",
        "avg_speed_percentage",
        "percent_slow_speed", "percent_boost_speed", "percent_supersonic_speed",
        "percent_ground", "percent_low_air", "percent_high_air",
    ]

    player_positioning = [
        "avg_distance_to_ball",
        "avg_distance_to_ball_possession",
        "avg_distance_to_ball_no_possession",
        "avg_distance_to_mates",
        "goals_against_while_last_defender",
        "percent_defensive_third",
        "percent_offensive_third",
        "percent_neutral_third",
        "percent_defensive_half",
        "percent_offensive_half",
        "percent_behind_ball",
        "percent_infront_ball",
        "percent_most_back",
        "percent_most_forward",
        "percent_closest_to_ball",
        "percent_farthest_from_ball",
    ]

    player_demo = ["inflicted", "taken"]

    for t in teams:
        for i in [0, 1]:
            p = f"{t}.players.{i}"
            keep.add(f"{p}.name")
            keep.add(f"{p}.id.platform")
            keep.add(f"{p}.steering_sensitivity")

            for f in player_core:
                keep.add(f"{p}.stats.core.{f}")
            for f in player_boost:
                keep.add(f"{p}.stats.boost.{f}")
            for f in player_movement:
                keep.add(f"{p}.stats.movement.{f}")
            for f in player_positioning:
                keep.add(f"{p}.stats.positioning.{f}")
            for f in player_demo:
                keep.add(f"{p}.stats.demo.{f}")

    return sorted(keep)


if "df_stage1" in globals():
    df = df_stage1.copy()
elif "df" in globals():
    print("[WARN] df_stage1 no existe. Usando el DataFrame actual 'df'.")
    df = df.copy()
else:
    raise NameError("df_stage1 no esta definido. Ejecuta la celda de Section 1 antes de Section 2.")
df = add_grouped_game_rank_columns(df)
keep_cols = build_keep_columns()

existing = [c for c in keep_cols if c in df.columns]
missing = [c for c in keep_cols if c not in df.columns]

print("========================================")
print("Section 2 - Seleccion de columnas")
print("========================================")
print(f"Filas actuales:      {len(df):,}")
print(f"Columnas actuales:   {df.shape[1]:,}")
print(f"Cols solicitadas:    {len(keep_cols):,}")
print(f"Cols encontradas:    {len(existing):,}")
print(f"Cols faltantes:      {len(missing):,}")

if missing:
    print("\nColumnas solicitadas que no existen:")
    for c in missing:
        print(f"- {c}")

df = df[existing].copy()
df_stage2 = df.copy()

print(f"\nColumnas tras filtro: {df.shape[1]:,}")

Section 2 - Seleccion de columnas
Filas actuales:      10,000
Columnas actuales:   535
Cols solicitadas:    272
Cols encontradas:    272
Cols faltantes:      0

Columnas tras filtro: 272


## Section 2.5 - Imputacion interactiva tras filtro

La imputacion interactiva se aplica despues de filtrar columnas para reducir entradas manuales innecesarias.

Esta seccion tambien guarda tus decisiones y, si ejecutas de nuevo, te permite preservar las decisiones anteriores o decidir otra vez.

In [13]:
df = df_stage2.copy()

# Robustez: permite ejecutar esta celda incluso si no se han recargado helpers/config
if "DECISIONS_25_FILE" not in globals():
    DECISIONS_25_FILE = "section_2_5_decisions.json"

if "_to_python_scalar" not in globals():
    def _to_python_scalar(value):
        if value is None:
            return None
        if pd.isna(value):
            return None
        if isinstance(value, np.generic):
            return value.item()
        return value

if "load_decisions" not in globals():
    def load_decisions(path: str) -> dict:
        if not os.path.exists(path):
            return {}
        try:
            with open(path, "r", encoding="utf-8") as f:
                data = json.load(f)
            return data if isinstance(data, dict) else {}
        except Exception:
            print(f"[WARN] No se pudo leer {path}. Se usaran decisiones vacias.")
            return {}

if "save_decisions" not in globals():
    def save_decisions(path: str, decisions: dict) -> None:
        serializable = {}
        for col, payload in decisions.items():
            if not isinstance(payload, dict):
                continue
            action = payload.get("action")
            if action == "drop":
                serializable[col] = {"action": "drop"}
            elif action == "impute":
                serializable[col] = {
                    "action": "impute",
                    "strategy": payload.get("strategy", "none"),
                    "value": _to_python_scalar(payload.get("value")),
                }
        with open(path, "w", encoding="utf-8") as f:
            json.dump(serializable, f, ensure_ascii=True, indent=2)

if "apply_saved_decisions" not in globals():
    def apply_saved_decisions(df_in: pd.DataFrame, decisions: dict) -> tuple[pd.DataFrame, list[str], dict[str, tuple[str, object]]]:
        dropped_cols = []
        imputed_cols = {}
        df_out = df_in.copy()

        for col, payload in decisions.items():
            if col not in df_out.columns:
                continue
            action = payload.get("action")

            if action == "drop":
                dropped_cols.append(col)
                continue

            if action == "impute":
                strat = payload.get("strategy", "none")
                val = payload.get("value")
                if strat != "none":
                    df_out[col] = df_out[col].fillna(val)
                preview = shorten(val, 60) if isinstance(val, str) else val
                imputed_cols[col] = (strat, preview)

        if dropped_cols:
            cols_to_drop = [c for c in dropped_cols if c in df_out.columns]
            df_out = df_out.drop(columns=cols_to_drop)

        return df_out, dropped_cols, imputed_cols

saved_decisions = load_decisions(DECISIONS_25_FILE)
use_saved = False

if saved_decisions:
    print("\n========================================")
    print("Decisiones previas detectadas")
    print("========================================")
    print(f"Archivo: {DECISIONS_25_FILE}")
    print(f"Columnas con decision guardada: {len(saved_decisions):,}")

    while True:
        choice = input("Quieres preservar decisiones previas o decidir otra vez? [p]=preservar, [r]=repetir: ").strip().lower()
        if choice in {"p", "r"}:
            use_saved = (choice == "p")
            break
        print("Entrada no valida. Usa p/r.")

if use_saved:
    df, dropped_cols, imputed_cols = apply_saved_decisions(df, saved_decisions)
    decisions = saved_decisions
    print("\nSe aplicaron las decisiones guardadas.")
else:
    null_ratio = df.isna().mean()
    keep_set = set(KEEP_COLS)

    candidates = [c for c in df.columns if (null_ratio[c] > THRESHOLD_NULL_RATIO and c not in keep_set)]
    candidates.sort(key=lambda c: null_ratio[c], reverse=True)

    dropped_cols = []
    imputed_cols = {}
    decisions = {}

    print("\n========================================")
    print("Candidatas a revisar (muchos nulos)")
    print(f"Total candidatas (null_ratio > {THRESHOLD_NULL_RATIO}): {len(candidates):,}")
    print("Formato: columna | null_pct | dtype")
    print("========================================")

    if not candidates:
        print("No hay columnas candidatas. Se continua a la siguiente seccion.")
    else:
        to_show = candidates if MAX_PRINT_CANDIDATES == 0 else candidates[:MAX_PRINT_CANDIDATES]
        for col in to_show:
            print(f"{col} | {null_ratio[col]*100:.2f}% | {df[col].dtype}")
        if MAX_PRINT_CANDIDATES != 0 and len(candidates) > MAX_PRINT_CANDIDATES:
            print(f"... ({len(candidates) - MAX_PRINT_CANDIDATES} mas no mostradas)")

        input("\nPulsa ENTER para empezar a decidir columna por columna...")

        for idx, col in enumerate(candidates, start=1):
            pct = float(null_ratio[col] * 100.0)
            dtype = str(df[col].dtype)

            print(f"\n[{idx}/{len(candidates)}]")

            while True:
                action = ask_action(col, pct, dtype)

                if action == "s":
                    print("\nEjemplos (no nulos):")
                    print_examples(df[col], n=SHOW_EXAMPLES)
                    continue

                if action == "q":
                    print("Salida solicitada. No se continua la ejecucion.")
                    raise SystemExit(0)

                if action == "d":
                    dropped_cols.append(col)
                    decisions[col] = {"action": "drop"}
                    break

                if action == "k":
                    if is_bool_dtype(df[col].dtype):
                        strat, val = ask_imputation_bool(col, df[col])
                    elif is_numeric_dtype(df[col].dtype):
                        strat, val = ask_imputation_numeric(col, df[col])
                    else:
                        strat, val = ask_imputation_categorical(col, df[col].astype("string"))

                    if strat != "none":
                        df[col] = df[col].fillna(val)

                    preview = val
                    if isinstance(preview, str):
                        preview = shorten(preview, 60)
                    imputed_cols[col] = (strat, preview)
                    decisions[col] = {
                        "action": "impute",
                        "strategy": strat,
                        "value": _to_python_scalar(val),
                    }
                    break

    if dropped_cols:
        df = df.drop(columns=dropped_cols)

    save_decisions(DECISIONS_25_FILE, decisions)
    print(f"\nDecisiones guardadas en: {DECISIONS_25_FILE}")

df_stage2_clean = df.copy()

print("\n========================================")
print("Section 2.5 completada (imputacion interactiva)")
print("========================================")
print(f"Filas actuales: {len(df):,}")
print(f"Columnas actuales: {df.shape[1]:,}")
print(f"Columnas eliminadas (interactivo): {len(dropped_cols):,}")
print(f"Columnas imputadas (rellenadas): {len(imputed_cols):,}")
print(f"Total nulls restantes: {int(df.isna().sum().sum()):,}")


Decisiones previas detectadas
Archivo: datasets/section_2_5_decisions.json
Columnas con decision guardada: 22

Se aplicaron las decisiones guardadas.

Section 2.5 completada (imputacion interactiva)
Filas actuales: 10,000
Columnas actuales: 272
Columnas eliminadas (interactivo): 0
Columnas imputadas (rellenadas): 22
Total nulls restantes: 125


## Section 3 - CambiosAVG (en memoria)

Se crean columnas *_pct a partir de columnas de tiempo, usando duration como denominador.

In [14]:
DURATION_COL = "duration"
TEAM_TIME_COLS = [
    "blue.stats.ball.possession_time",
    "blue.stats.ball.time_in_side",
    "orange.stats.ball.possession_time",
    "orange.stats.ball.time_in_side",
]

PLAYER_TIME_SUFFIXES = [
    "time_supersonic_speed",
    "time_boost_speed",
    "time_slow_speed",
    "time_ground",
    "time_low_air",
    "time_high_air",
    "time_powerslide",
]


def add_pct_column(df_in: pd.DataFrame, time_col: str, duration_col: str) -> bool:
    if time_col not in df_in.columns:
        print(f"[WARN] No existe: {time_col}")
        return False
    if duration_col not in df_in.columns:
        raise KeyError(f"No existe la columna de duracion: {duration_col}")

    t = to_numeric(df_in[time_col])
    d = to_numeric(df_in[duration_col])

    pct = (t / d) * 100.0
    pct = pct.where(d > 0)

    df_in[f"{time_col}_pct"] = pct
    return True


df = df_stage2_clean.copy()
created_pct_cols = []

for col in TEAM_TIME_COLS:
    if add_pct_column(df, col, DURATION_COL):
        created_pct_cols.append(f"{col}_pct")

for team in ["blue", "orange"]:
    for i in [0, 1]:
        base = f"{team}.players.{i}.stats.movement"
        for suf in PLAYER_TIME_SUFFIXES:
            col = f"{base}.{suf}"
            if add_pct_column(df, col, DURATION_COL):
                created_pct_cols.append(f"{col}_pct")

df_stage3 = df.copy()

print("========================================")
print("Section 3 - Conversion a porcentajes")
print("========================================")
print(f"Columnas _pct creadas: {len(created_pct_cols):,}")
if created_pct_cols:
    print("Primeras 10 columnas creadas:")
    for c in created_pct_cols[:10]:
        print(f"- {c}")

Section 3 - Conversion a porcentajes
Columnas _pct creadas: 32
Primeras 10 columnas creadas:
- blue.stats.ball.possession_time_pct
- blue.stats.ball.time_in_side_pct
- orange.stats.ball.possession_time_pct
- orange.stats.ball.time_in_side_pct
- blue.players.0.stats.movement.time_supersonic_speed_pct
- blue.players.0.stats.movement.time_boost_speed_pct
- blue.players.0.stats.movement.time_slow_speed_pct
- blue.players.0.stats.movement.time_ground_pct
- blue.players.0.stats.movement.time_low_air_pct
- blue.players.0.stats.movement.time_high_air_pct


## Final - Validacion y export

Solo se guarda el CSV final del pipeline.

In [15]:
df_final = df_stage3.copy()

pct_cols = [c for c in df_final.columns if c.endswith("_pct")]

print("========================================")
print("Resumen final")
print("========================================")
print(f"Filas finales:      {len(df_final):,}")
print(f"Columnas finales:   {df_final.shape[1]:,}")
print(f"Total nulls final:  {int(df_final.isna().sum().sum()):,}")
print(f"Columnas _pct:      {len(pct_cols):,}")

if pct_cols:
    desc = df_final[pct_cols].describe().T[["mean", "min", "max"]]
    print("\nResumen de columnas _pct (primeras 15):")
    display(desc.head(15))

# Unica exportacion del flujo unificado
print(f"\nGuardando CSV final en: {CSV_OUT}")
df_final.to_csv(CSV_OUT, index=False)

print("\nProceso completado.")

Resumen final
Filas finales:      10,000
Columnas finales:   304
Total nulls final:  139
Columnas _pct:      32

Resumen de columnas _pct (primeras 15):


,mean,min,max
blue.stats.ball.possession_time_pct,37.540408,0.0,66.601562
blue.stats.ball.time_in_side_pct,45.519349,0.0,113.500000
orange.stats.ball.possession_time_pct,37.637201,0.0,64.565657
orange.stats.ball.time_in_side_pct,45.854716,0.0,81.652174
blue.players.0.stats.movement.time_supersonic_speed_pct,13.872140,0.0,69.923077
blue.players.0.stats.movement.time_boost_speed_pct,38.162868,0.0,58.295455
blue.players.0.stats.movement.time_slow_speed_pct,47.015995,0.0,90.000000
blue.players.0.stats.movement.time_ground_pct,55.896762,0.0,88.625000
blue.players.0.stats.movement.time_low_air_pct,38.468076,0.0,77.692308
blue.players.0.stats.movement.time_high_air_pct,4.686183,0.0,20.815166



Guardando CSV final en: datasets/replays_subset_with_time_percentages.csv

Proceso completado.
